# Atividade Aula 3

Regressão Logística utilizando PyTorch e o dataset Student Exam Performance Prediction.

## Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import kagglehub


## Download do dataset

In [ ]:
# Download do dataset
path = kagglehub.dataset_download("mrsimple07/student-exam-performance-prediction")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'student-exam-performance-prediction' dataset.
Path to dataset files: /kaggle/input/student-exam-performance-prediction


## Leitura dos dados

In [ ]:
import os

files = os.listdir(path)
print(files)

csv_file = [f for f in files if f.endswith('.csv')][0]

df = pd.read_csv(os.path.join(path, csv_file))

df.head()


['student_exam_data_new.csv', 'student_exam_data.csv']


,Study Hours,Previous Exam Score,Pass/Fail
0,4.370861,81.889703,0
1,9.556429,72.165782,1
2,7.587945,58.571657,0
3,6.387926,88.827701,1
4,2.404168,81.083870,0


## Pré-processamento

In [ ]:
# Converter colunas categóricas
label_encoders = {}

for column in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column])
    label_encoders[column] = le

df.head()


,Study Hours,Previous Exam Score,Pass/Fail
0,4.370861,81.889703,0
1,9.556429,72.165782,1
2,7.587945,58.571657,0
3,6.387926,88.827701,1
4,2.404168,81.083870,0


In [ ]:
# Definir variável alvo
target_column = df.columns[-1]

X = df.drop(target_column, axis=1)
y = df[target_column]

# Normalização
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train/test split
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Converter para tensor
x_train = torch.FloatTensor(x_train)
x_test = torch.FloatTensor(x_test)

y_train = torch.LongTensor(y_train.values)
y_test = torch.LongTensor(y_test.values)

print(x_train.shape)
print(y_train.shape)


torch.Size([400, 2])
torch.Size([400])


## Criação da rede

In [ ]:
input_size = x_train.shape[1]
output_size = len(torch.unique(y_train))

# Regressão logística simples
model = nn.Linear(input_size, output_size)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)


Linear(in_features=2, out_features=2, bias=True)


## Treinamento

In [ ]:
epochs = 500

for epoch in range(epochs):

    # Forward
    outputs = model(x_train)

    loss = criterion(outputs, y_train)

    # Backward
    optimizer.zero_grad()
    loss.backward()

    # Update
    optimizer.step()

    if (epoch+1) % 20 == 0:
        _, predicted = torch.max(outputs, 1)

        accuracy = (predicted == y_train).float().mean()

        print(f'Época [{epoch+1}/{epochs}] - Loss: {loss.item():.4f} - Accuracy: {accuracy.item():.4f}')


Época [20/500] - Loss: 0.5350 - Accuracy: 0.6775
Época [40/500] - Loss: 0.5255 - Accuracy: 0.6825
Época [60/500] - Loss: 0.5163 - Accuracy: 0.6875
Época [80/500] - Loss: 0.5076 - Accuracy: 0.6975
Época [100/500] - Loss: 0.4992 - Accuracy: 0.7050
Época [120/500] - Loss: 0.4911 - Accuracy: 0.7075
Época [140/500] - Loss: 0.4834 - Accuracy: 0.7075
Época [160/500] - Loss: 0.4760 - Accuracy: 0.7200
Época [180/500] - Loss: 0.4689 - Accuracy: 0.7325
Época [200/500] - Loss: 0.4621 - Accuracy: 0.7375
Época [220/500] - Loss: 0.4555 - Accuracy: 0.7475
Época [240/500] - Loss: 0.4492 - Accuracy: 0.7525
Época [260/500] - Loss: 0.4432 - Accuracy: 0.7600
Época [280/500] - Loss: 0.4373 - Accuracy: 0.7625
Época [300/500] - Loss: 0.4317 - Accuracy: 0.7725
Época [320/500] - Loss: 0.4263 - Accuracy: 0.7800
Época [340/500] - Loss: 0.4211 - Accuracy: 0.7850
Época [360/500] - Loss: 0.4160 - Accuracy: 0.7925
Época [380/500] - Loss: 0.4112 - Accuracy: 0.7975
Época [400/500] - Loss: 0.4065 - Accuracy: 0.8075
Époc

## Avaliação

In [ ]:
with torch.no_grad():
    outputs = model(x_test)
    _, predicted = torch.max(outputs, 1)

accuracy = accuracy_score(y_test.numpy(), predicted.numpy())

print(f'Acurácia no conjunto de teste: {accuracy:.4f}')


Acurácia no conjunto de teste: 0.8000


# Aprendizados

Nesta atividade foi possível aplicar Regressão Logística utilizando PyTorch em um dataset do Kaggle. Também foi possível praticar:
pré-processamento de dados;
normalização;
uso do Linear do PyTorch;
treinamento utilizando SGD;
cálculo de acurácia.

Ao aumentar o número de épocas de 300 para 500, a acurácia melhorou significativamente. Isso mostrou que o modelo ainda estava aprendendo e precisava de mais iterações para ajustar os pesos corretamente. Mesmo utilizando uma rede simples com apenas uma camada linear, foi possível obter resultados melhores apenas ajustando os hiperparâmetros do treinamento.
